# Phase 1: Sentinel-2 Burned Area Extraction (dNBR)

**Methodology:** Based on the 2023 SPIE publication: *"Harvesting remote sensing observations for quantifying burned area and built-up losses from the 2021 wildfires in Greece."*

This notebook automates the QGIS spatial analysis workflow using Python and `rasterio`. It loads pre- and post-fire Copernicus Sentinel-2A imagery, calculates the Normalized Burn Ratio (NBR) using bands 8A and 12, computes the differenced NBR (dNBR), and generates a binary raster mask of the burned area.

In [ ]:
# --- 1. Imports and Environment Setup ---
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import show
from matplotlib.colors import ListedColormap

# Suppress scientific notation for cleaner outputs
np.set_printoptions(suppress=True)

print("Geospatial libraries loaded successfully.")

## 2. Configuration & File Paths
All input paths (satellite bands and AOI vector file) and output paths are configured here. The AOI file can be a GeoJSON, Shapefile, or GeoPackage.

In [ ]:
# --- Configuration & File Paths ---
DATA_DIR = "data"

# Area of Interest (AOI) boundary file (GeoJSON, SHP, or GPKG)
AOI_PATH = os.path.join(DATA_DIR, "aoi_boundary.geojson")

# Raw Sentinel-2 Bands (20m resolution)
PRE_FIRE_B8A = os.path.join(DATA_DIR, "pre_fire_b8a.tif")
PRE_FIRE_B12 = os.path.join(DATA_DIR, "pre_fire_b12.tif")

POST_FIRE_B8A = os.path.join(DATA_DIR, "post_fire_b8a.tif")
POST_FIRE_B12 = os.path.join(DATA_DIR, "post_fire_b12.tif")

# Export Outputs
OUTPUT_DNBR = os.path.join(DATA_DIR, "output_dnbr_aoi.tif")
OUTPUT_BURN_MASK = os.path.join(DATA_DIR, "output_burn_mask_aoi.tif")

print("Paths configured.")

## 3. AOI Loading & Cropped Raster Ingestion
To crop cleanly, we read the AOI polygon using `geopandas`. The helper function reprojects the polygon on the fly to match the raster's CRS, then uses `rasterio.mask.mask` to extract only the pixels inside the polygon and update the bounding box and affine transform.

In [ ]:
# Load Area of Interest
aoi_gdf = gpd.read_file(AOI_PATH)
print(f"AOI loaded: {len(aoi_gdf)} polygon(s) found.")

def read_and_crop_band(raster_path, aoi):
    """
    Opens a raster, aligns the AOI CRS, crops the raster to the polygon boundary,
    and returns the 2D array along with the updated spatial profile.
    """
    with rasterio.open(raster_path) as src:
        # Reproject AOI to match raster CRS if necessary
        if aoi.crs != src.crs:
            aoi = aoi.to_crs(src.crs)
            
        # Crop raster to AOI geometry
        shapes = [geom for geom in aoi.geometry]
        cropped_image, cropped_transform = mask(src, shapes=shapes, crop=True)
        
        # Extract band 1 as float32
        band = cropped_image[0].astype('float32')
        
        # Update metadata profile with new cropped dimensions and transform
        profile = src.profile.copy()
        profile.update({
            "height": band.shape[0],
            "width": band.shape[1],
            "transform": cropped_transform
        })
        
    return band, profile

# Crop and load pre-fire imagery
pre_b8a, cropped_profile = read_and_crop_band(PRE_FIRE_B8A, aoi_gdf)
pre_b12, _ = read_and_crop_band(PRE_FIRE_B12, aoi_gdf)

# Crop and load post-fire imagery
post_b8a, _ = read_and_crop_band(POST_FIRE_B8A, aoi_gdf)
post_b12, _ = read_and_crop_band(POST_FIRE_B12, aoi_gdf)

print(f"Bands cropped successfully. AOI raster dimensions: {pre_b8a.shape}")

## 4. Spectral Index Calculation (NBR & dNBR)
The Normalized Burn Ratio (NBR) is calculated across bands 8A (VNIR) and 12 (SWIR)[cite: 2]:
$$\text{NBR} = \frac{B8A - B12}{B8A + B12}$$

Differenced NBR (dNBR) is the change in reflectance between pre-fire and post-fire states[cite: 2]:
$$\text{dNBR} = \text{NBR}_{\text{pre}} - \text{NBR}_{\text{post}}$$

In [ ]:
def calculate_nbr(b8a, b12):
    """Calculates Normalized Burn Ratio with zero-division handling."""
    with np.errstate(divide='ignore', invalid='ignore'):
        nbr = (b8a - b12) / (b8a + b12)
    return np.nan_to_num(nbr, nan=0.0)

# Calculate pre- and post-fire NBR
pre_nbr = calculate_nbr(pre_b8a, pre_b12)
post_nbr = calculate_nbr(post_b8a, post_b12)

# Calculate dNBR
dnbr = pre_nbr - post_nbr

print(f"dNBR calculated inside AOI.")
print(f"Min: {np.min(dnbr):.3f} | Max: {np.max(dnbr):.3f} | Mean: {np.mean(dnbr):.3f}")

## 5. Burn Severity Thresholding & Area Estimation
Grid cells with $\text{dNBR} \ge 0.1$ are classified as burned, while values $< 0.1$ are considered unburned[cite: 2]. The total burned surface area is computed based on Sentinel-2's 20m pixel resolution ($400\,\text{m}^2$ per pixel).

In [ ]:
# Create binary mask (1 = Burned, 0 = Unburned)
burn_mask = np.where(dnbr >= 0.1, 1, 0).astype('uint8')

# Calculate area
burned_pixel_count = np.sum(burn_mask == 1)
pixel_area_sqm = 20 * 20  # 400 sqm
total_burned_sqkm = (burned_pixel_count * pixel_area_sqm) / 1_000_000

print(f"Total Burned Area within AOI: {total_burned_sqkm:.2f} km²")

## 6. Exporting Cropped Rasters
The clipped continuous dNBR and binary burn mask are written to GeoTIFF format, preserving the AOI's bounding box and affine georeference.

In [ ]:
# Export continuous dNBR raster
dnbr_profile = cropped_profile.copy()
dnbr_profile.update(dtype=rasterio.float32, count=1, nodata=-9999.0)

with rasterio.open(OUTPUT_DNBR, 'w', **dnbr_profile) as dst:
    dst.write(dnbr, 1)

# Export binary burn scar mask
mask_profile = cropped_profile.copy()
mask_profile.update(dtype=rasterio.uint8, count=1, nodata=0)

with rasterio.open(OUTPUT_BURN_MASK, 'w', **mask_profile) as dst:
    dst.write(burn_mask, 1)

print(f"Rasters exported:\n - {OUTPUT_DNBR}\n - {OUTPUT_BURN_MASK}")

## 7. Results Visualization
Plotting the pre-fire NBR, post-fire NBR, differenced dNBR, and the resulting thresholded burn scar within the AOI boundary.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Pre-fire NBR
im0 = axes[0].imshow(pre_nbr, cmap='YlGn')
axes[0].set_title("Pre-Fire NBR")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# 2. dNBR
im1 = axes[1].imshow(dnbr, cmap='RdYlGn_r', vmin=-0.2, vmax=0.8)
axes[1].set_title("Differenced NBR (dNBR)")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# 3. Burn Scar Mask
cmap_mask = ListedColormap(['#e0e0e0', '#d73027'])
axes[2].imshow(burn_mask, cmap=cmap_mask)
axes[2].set_title(f"Burn Scar Mask (≥ 0.1)\nArea: {total_burned_sqkm:.2f} km²")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()